# 第十二课｜一个 spike 的完整旅程

前三课分别解决了：很多 state 如何共享 engine、spike 如何排队、source 如何找到真实下游连接。今天不再引入一个大型新模块，而是把它们接起来：
> **一个 source spike 从 queue 出发，怎样变成 target update，并可能产生新的 spike？**

主要新概念：**事件驱动计算（event-driven computation）** 的端到端因果链。


## 1. 概念账本

**已经知道：** time multiplexing、FIFO/backpressure、source_index + sparse synapse records。

**今天学习：** 把已有概念组成一条 event-driven pipeline；支持术语包括 router、weighted event、synapse stream。

**只预告：** 正式 RTL 的 valid/ready streaming、并发冲突和高性能 target accumulator 以后再实现。


## 2. event-driven 到底是什么意思？

不是每个 cycle 扫描所有可能连接，而是**只有发生 spike 时，才处理这个 source 对应的真实下游 synapse**。

这改变的是计算组织方式，不是 neuron 生物学模型本身。


## 3. 一条完整路径

```mermaid
flowchart LR
 Q["spike FIFO"] --> S["source_id"]
 S --> IDX["source index lookup"]
 IDX --> R["synapse records"]
 R --> W["weighted events: target, weight"]
 W --> A["target accumulator / update"]
 A --> T["threshold rule"]
 T -->|new spike| Q
```

可以把 **router** 理解成“根据 source_id 决定 event 下一步去哪里”的控制/查找部分；**synapse stream** 是依次产生的 `(source,target,weight,...)` 记录流。


## 4. 四神经元手工网络

为了只观察事件旅程，本课使用一个极小的教学 threshold accumulator，而不是正式 LIF：

- 0 → 1，weight +2
- 0 → 2，weight +1
- 1 → 3，weight +2
- 2 → 3，weight +1

threshold：neuron 1=2，2=1，3=3。初始只有 neuron 0 spike。

先手工预测 spike order。


In [ ]:
from collections import deque

# Four-neuron teaching network.
# records for source 0: (1,+2), (2,+1)
# source 1: (3,+2), source 2: (3,+1), source 3: none
source_index = [(0, 2), (2, 1), (3, 1), (4, 0)]
records = [(1, 2), (2, 1), (3, 2), (3, 1)]
threshold = [99, 2, 1, 3]
accum = [0, 0, 0, 0]
queue = deque([0])
spike_order = []

while queue:
    source = queue.popleft()
    spike_order.append(source)
    print(f'\nconsume spike source={source}')
    start, count = source_index[source]
    for target, weight in records[start:start+count]:
        accum[target] += weight
        print(f'  weighted event -> target={target}, weight={weight}, accum={accum[target]}')
        if accum[target] >= threshold[target]:
            print(f'  target {target} spikes; enqueue it and reset teaching accumulator')
            accum[target] = 0
            queue.append(target)

print('\nspike order:', spike_order)
print('final accum:', accum)


## 5. Observe：不要只看最终结果

逐条追踪因果：

1. source 0 出队；
2. lookup 得到两条真实 synapse；
3. target 1/2 收到 weighted event 并达到 threshold；
4. 1 和 2 进入 queue；
5. 它们随后各自给 target 3 贡献输入；
6. target 3 累积到 3 后产生新 spike。

预期 spike order 是 `[0, 1, 2, 3]`。


## 6. 一个非常重要的边界

这段 Python 是**教学 event machine**，不是正式 neuron model：

- threshold accumulator 被刻意简化；
- 没有 leak、fixed-point、refractory；
- 没有并发 target write conflict；
- 没有 valid/ready timing。

因此不能把它的数值语义复制进正式 `MOD-003`。本课只验证 event routing 的因果结构。


## 7. Try It：删一条 edge

先预测：若删掉 `2 → 3, +1`，neuron 3 还会不会 spike？为什么？

再从 `records` 中删掉对应记录，并同步修改 source_index。这个实验也会暴露为什么 sparse index 与 records 必须保持一致。


## 8. 作业

完成 `exercises/lesson12_event_journey.py` 的 `process_one_spike(...)`：输入一个 source event，只处理它对应的 synapse range，返回 weighted events 与更新后的 target accumulators。

```bash
uv run pytest exercises/checks/check_lesson12.py -q
```


## 9. AI Task

把一次失败的 event trace 交给 AI，要求它按 `queue → source lookup → synapse record → target accumulation` 四层定位第一处因果不一致；不允许它一开始就改 neuron threshold。


## 10. Human Check

不用 AI，你应该能从 source 0 手工推到 `[0,1,2,3]`；指出 queue 保存什么、source_index 决定什么、synapse record 携带什么、target accumulator 改变什么；解释为什么 event-driven 不等于“神经元只在 spike 时才拥有 state”。


## 11. Engineering Handoff

本课对应 RMD-007A 的四神经元 walk-through，并为 RMD-010 synapse reader + engine、RMD-011 small event-driven SNN 建立共同语义。正式 `MOD-005~009`、T-010~T-013 仍需逐模块实现和验证。


## 12. 项目追踪 Project Trace

- Lesson: `LSN-012`
- Mapping: `RMD-007A / RMD-010 / RMD-011` teaching integration
- Module context: `MOD-005~009`
- Test context: prepares `T-010~T-013`


## 13. Exit Ticket

你能不用代码讲清楚一个 spike 从 source event 到 target update，再到新 spike 入队的完整路径，并知道每一段应由哪类测试负责。
